# 82514 · Sesión S21 — TF2 en numpy: el árbol de marcos del almacén

**Bloque 7** · lunes 30 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** Rehace en numpy lo que hace TF2 con el árbol de marcos del almacén del curso, con el mismo código del paquete `curso_almacen` que corre en el simulador del taller: las transformadas y el cuaternión en el orden de ROS; el árbol map → odom → base_link → laser (REP 105) y el del puesto de picking, con la cámara de S16 y la pieza de S17; la predicción de lo que dirá `tf2_echo map laser`; la diferencia entre ir a un punto controlando en odom y en map; y por qué se pregunta por la pose en el instante de la medida. No ejecuta ROS 2: el taller se hace en el portátil.

**Se apoya en:** la documentación de tf2 de ROS 2 Jazzy; REP 105, la convención de marcos de un robot móvil; Foote (2013), el artículo de tf; Corke (2023), ir a un punto; las transformadas homogéneas y el cuaternión de S8; el paquete curso_almacen del curso.

**Cómo usarlo en clase.** Sigue el guion de la sesión S21 en las diapositivas del bloque 7. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos: no instala nada.
import sys, os, json, math, time
import numpy as np, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

FIJAS = {'numpy': '2.4.6', 'matplotlib': '3.10.0'}          # las de Colab con las que se probó el bloque 7
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

for nombre, mod in (('numpy', np), ('matplotlib', matplotlib)):
    print(nombre, mod.__version__, '' if mod.__version__ == FIJAS[nombre] else f'(probado con {FIJAS[nombre]})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

> **Aviso para Colab.** Este cuaderno no ejecuta ROS 2: ROS 2 corre en vuestro portátil, el de la comprobación de S20. Aquí se rehace en numpy lo que hace TF2, con el mismo código del simulador del taller. No instala nada (usa numpy y matplotlib de Colab) y la primera celda tarda unos segundos.
>
> **En clase (lunes 30, en la ronda 2, unos dos minutos):** ejecuta las secciones 0 a 2 sin detenerte y haz la sección 3: con el robot parado, copia lo que dicen dos `tf2_echo` y compara la predicción con el tercero.
>
> **En casa:** las secciones 4 a 6 y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** el entregable del taller es el zip del paquete `taller_s21`; este cuaderno no tiene tarea en Moodle.

## 0. El almacén del curso, con el código del simulador

*En clase, ejecuta esta celda sin detenerte.*

Es el fichero `almacen.py` del paquete `curso_almacen`, tal cual: el mismo que usa el nodo `simulador` del taller. El almacén de B6 (13 × 9 m en celdas de 10 cm, dos naves y un muro con una puerta ancha y una estrecha, y cinco estanterías), los puntos con nombre (el muelle en (0,6; 0,6) y el picking en (8,3; 8,2)), el robot (r = 0,28 m, 0,5 m/s y 1,0 rad/s), el montaje del LiDAR (base_link → laser = (0,10; 0; 0,20) m), la odometría de B6 (tras un metro, 6 cm y 0,9°, que crecen con la raíz de los metros) y el LiDAR de S6 (1440 haces, 2 cm de ruido, un 2 % sin retorno y 10 m de alcance).

In [ ]:
# El almacén del curso: curso_almacen/almacen.py, el mismo código que el simulador de ROS 2 del taller.
import math

import numpy as np

RES, FILAS, COLS = 0.10, 90, 130          # celdas de 10 cm: fila = y/0,1 y columna = x/0,1
R_ROBOT = 0.28                            # el radio del robot del almacén (m)
V_MAX, W_MAX = 0.5, 1.0                   # sus límites: 0,5 m/s y 1,0 rad/s
LASER = (0.10, 0.0, 0.20)                 # el montaje del LiDAR: base_link -> laser (m)
SIG_D, SIG_T = 0.06, math.radians(0.9)    # la odometría de B6: 6 cm y 0,9° por raíz de metro recorrido
N_HACES = 1440                            # el LiDAR de S6: 360° a 0,25°
INC = math.radians(0.25)
ANG_MIN = -math.pi                        # el haz 720 mira hacia delante (0°)
RMAX, SIG_L, P_SIN = 10.0, 0.02, 0.02     # alcance de 10 m, 2 cm de ruido y un 2 % de rayos sin retorno

MUELLE, PICKING, DERECHA = (0.6, 0.6), (8.3, 8.2), (11.5, 6.0)   # los puntos con nombre (m)
PUERTA_ESTRECHA = (slice(42, 47), slice(92, 100))                # sus celdas: y de 4,2 a 4,7 y x de 9,2 a 10,0


def mapa_almacen(puerta_estrecha='abierta'):
    """La rejilla de ocupación del almacén (True = ocupado)."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # las paredes
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # las estanterías de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # las de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    if puerta_estrecha == 'cerrada':
        occ[PUERTA_ESTRECHA] = True
    return occ


def distancias(occ):
    """La distancia de cada celda (su centro) al obstáculo más cercano (el borde de su celda), en m.
    Se calcula una vez al arrancar, a fuerza bruta (menos de un segundo)."""
    oc = np.argwhere(occ)
    cy, cx = (oc[:, 0] + 0.5) * RES, (oc[:, 1] + 0.5) * RES
    gy, gx = np.mgrid[0:FILAS, 0:COLS]
    px, py = ((gx + 0.5) * RES).ravel(), ((gy + 0.5) * RES).ravel()
    d = np.empty(px.size)
    for k in range(0, px.size, 2000):
        dx = np.maximum(np.abs(px[k:k + 2000, None] - cx[None]) - RES / 2, 0)
        dy = np.maximum(np.abs(py[k:k + 2000, None] - cy[None]) - RES / 2, 0)
        d[k:k + 2000] = np.hypot(dx, dy).min(axis=1)
    return d.reshape(FILAS, COLS)


class Mundo:
    """El almacén físico: lo que ven el LiDAR y los choques (con la puerta estrecha abierta o cerrada)."""

    def __init__(self, puerta_estrecha='abierta'):
        self.occ = mapa_almacen(puerta_estrecha)
        self.dist = distancias(self.occ)
        oc = np.argwhere(self.occ)
        self.cy, self.cx = (oc[:, 0] + 0.5) * RES, (oc[:, 1] + 0.5) * RES
        self.ang = ANG_MIN + INC * np.arange(N_HACES)

    def holgura(self, x, y):
        """La distancia del punto (x; y) al obstáculo más cercano, en m."""
        dx = np.maximum(np.abs(x - self.cx) - RES / 2, 0)
        dy = np.maximum(np.abs(y - self.cy) - RES / 2, 0)
        return float(np.hypot(dx, dy).min())

    def rayos(self, x, y, th):
        """La distancia al primer obstáculo de cada uno de los 1440 haces, desde (x; y) con rumbo th.
        Avanza cada rayo lo que deja libre la tabla de distancias, y a pasos de 1 cm cerca de una pared;
        un rayo que no encuentra nada en 10 m devuelve inf."""
        a = th + self.ang
        ca, sa = np.cos(a), np.sin(a)
        t = np.full(a.size, 0.01)
        vivo = np.ones(a.size, bool)
        for _ in range(400):
            px, py = x + t * ca, y + t * sa
            cc, ff = np.floor(px / RES).astype(int), np.floor(py / RES).astype(int)
            fuera = (cc < 0) | (cc >= COLS) | (ff < 0) | (ff >= FILAS)
            cc, ff = np.clip(cc, 0, COLS - 1), np.clip(ff, 0, FILAS - 1)
            vivo &= ~(fuera | self.occ[ff, cc]) & (t < RMAX)
            if not vivo.any():
                break
            paso = np.maximum(self.dist[ff, cc] - 0.0708, 0.01)   # 0,0708 = media diagonal de una celda
            t = np.where(vivo, t + paso, t)
        return np.where(t >= RMAX, np.inf, t)

    def barrido(self, x, y, th, rng, ruido=True):
        """Un barrido del LiDAR desde el marco laser, con su ruido y sus rayos perdidos (inf)."""
        r = self.rayos(x, y, th)
        if ruido:
            r = r + rng.normal(0, SIG_L, r.size)
            r[rng.random(r.size) < P_SIN] = np.inf
        return r

## 1. Transformadas y el cuaternión en el orden de ROS

*En clase, ejecuta estas dos celdas sin detenerte.*

**T_AB es la pose de {B} vista desde {A}**, la transformada homogénea 4 × 4 de S8; en TF2 se escribe «A → B», del padre al hijo. Su inversa es la de S8, [Rᵀ, −Rᵀ·t; 0, 1], y dos transformadas seguidas se componen multiplicando: T_AC = T_AB · T_BC, la letra de en medio se cancela.

**El cuaternión de ROS es el de S8 en otro orden:** (x, y, z, w) = (v; s), primero la parte vectorial y al final el escalar. Un giro θ alrededor de z da (0; 0; sen θ/2; cos θ/2), y de vuelta θ = 2·atan2(z, w). `como_tf2_echo()` escribe una transformada como lo hace `ros2 run tf2_ros tf2_echo`: la traslación y el cuaternión (xyzw), con tres decimales.

La clase `Arbol` guarda, para cada marco, su padre y la transformada padre → hijo, que es lo que viaja por `/tf` y `/tf_static`. `lookup(destino, origen)` hace lo que `lookup_transform`: sube de cada marco a la raíz, acumulando, y da T_destino,origen. Si los dos marcos no comparten raíz, no hay camino y falla.

In [ ]:
# Las transformadas, el cuaternión (x, y, z, w) y el árbol de marcos: el mismo código que las cifras de las diapositivas.
# =====================================================================================================
# S21 · El árbol de transformadas en numpy: T_AB es la pose de {B} vista desde {A} (S8), «A -> B» en TF2
# =====================================================================================================
def Rz(a):
    c, s = math.cos(a), math.sin(a)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])


def Rx(a):
    c, s = math.cos(a), math.sin(a)
    return np.array([[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]])


def T(R=np.eye(3), t=(0.0, 0.0, 0.0)):
    """La transformada homogénea 4 x 4 con la rotación R y la traslación t."""
    M = np.eye(4)
    M[:3, :3], M[:3, 3] = R, t
    return M


def inv(M):
    """La inversa de S8: [Rᵀ, −Rᵀ·t; 0, 1]."""
    Ri = M[:3, :3].T
    return T(Ri, -Ri @ M[:3, 3])


def plana(x, y, th, z=0.0):
    """Una pose del plano (x; y; θ) como transformada 4 x 4."""
    return T(Rz(th), (x, y, z))


def cuaternion(R):
    """El cuaternión de R en el orden de ROS: (x, y, z, w) = (v; s), con w ≥ 0 (en S8 era (s; v)). Sale de la mayor de
    sus cuatro componentes, para no dividir por una pequeña, y se normaliza: así es unitario aunque R venga redondeada,
    como la T_C0 del JSON de S16, que gira casi 180°."""
    t = [1 + R[0, 0] + R[1, 1] + R[2, 2], 1 + R[0, 0] - R[1, 1] - R[2, 2],
         1 - R[0, 0] + R[1, 1] - R[2, 2], 1 - R[0, 0] - R[1, 1] + R[2, 2]]
    k = int(np.argmax(t))
    s = 2 * math.sqrt(max(t[k], 0.0))
    if k == 0:
        q = [(R[2, 1] - R[1, 2]) / s, (R[0, 2] - R[2, 0]) / s, (R[1, 0] - R[0, 1]) / s, s / 4]
    elif k == 1:
        q = [s / 4, (R[0, 1] + R[1, 0]) / s, (R[0, 2] + R[2, 0]) / s, (R[2, 1] - R[1, 2]) / s]
    elif k == 2:
        q = [(R[0, 1] + R[1, 0]) / s, s / 4, (R[1, 2] + R[2, 1]) / s, (R[0, 2] - R[2, 0]) / s]
    else:
        q = [(R[0, 2] + R[2, 0]) / s, (R[1, 2] + R[2, 1]) / s, s / 4, (R[1, 0] - R[0, 1]) / s]
    q = np.array(q) / np.linalg.norm(q)
    q[np.abs(q) < 1e-12] = 0.0
    if q[3] < 0 or (q[3] == 0 and q[np.flatnonzero(q)[0]] < 0):    # w ≥ 0; con w = 0 (180°), la primera no nula, positiva
        q = -q
    return q + 0.0


def guinada(q):
    """El rumbo de un cuaternión (x, y, z, w) que solo gira en z: θ = 2·atan2(z, w)."""
    return 2 * math.atan2(q[2], q[3])


def coma(x, n=3):
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')


def como_tf2_echo(M):
    """Lo que imprimiría ros2 run tf2_ros tf2_echo: la traslación y el cuaternión (x, y, z, w), con tres decimales."""
    q = cuaternion(M[:3, :3])
    return (f'- Translation: [{M[0, 3]:.3f}, {M[1, 3]:.3f}, {M[2, 3]:.3f}]\n'
            f'- Rotation: in Quaternion (xyzw) [{q[0]:.3f}, {q[1]:.3f}, {q[2]:.3f}, {q[3]:.3f}]')


def describe(M):
    q = cuaternion(M[:3, :3])
    th = math.degrees(math.atan2(M[1, 0], M[0, 0]))
    return (f'({coma(M[0, 3])}; {coma(M[1, 3])}; {coma(M[2, 3])}) m, guiñada {coma(th, 1)}°, '
            f'(x, y, z, w) = ({"; ".join(coma(v, 4) for v in q)})')


class Arbol:
    """Un árbol de marcos como el de TF2: cada marco tiene un padre y la transformada padre -> hijo."""

    def __init__(self):
        self.padre = {}

    def pon(self, padre, hijo, M):
        self.padre[hijo] = (padre, M)

    def raiz_de(self, marco):
        """La cadena de un marco hasta su raíz: [(marco, T_raiz,marco)], acumulando de la raíz hacia abajo."""
        cadena, M = [marco], np.eye(4)
        while cadena[-1] in self.padre:
            p, Tp = self.padre[cadena[-1]]
            M = Tp @ M
            cadena.append(p)
        return cadena[-1], M

    def lookup(self, destino, origen):
        """Como lookup_transform(destino, origen): la pose de origen vista desde destino, T_destino,origen."""
        r1, M1 = self.raiz_de(destino)
        r2, M2 = self.raiz_de(origen)
        if r1 != r2:
            raise LookupError(f'{destino} y {origen} no están en el mismo árbol (raíces {r1} y {r2})')
        return inv(M1) @ M2


def arbol_almacen(robot_map=(8.3, 8.2, math.pi / 2), map_odom=(0.30, -0.15, math.radians(2.0))):
    """El árbol del almacén y del puesto (REP 105), con el robot en robot_map (la foto: aparcado en el picking)."""
    a = Arbol()
    Tmo = plana(*map_odom)
    a.pon('map', 'odom', Tmo)                                            # la localización: a saltos
    a.pon('odom', 'base_link', inv(Tmo) @ plana(*robot_map))             # la odometría: continua, con deriva
    a.pon('base_link', 'laser', T(np.eye(3), LASER))                     # fija: el montaje del LiDAR
    a.pon('map', 'mesa', T(Rz(-math.pi / 2), (8.3, 8.9, 0.75)))          # fija: el puesto de picking
    a.pon('mesa', 'camara', T(Rx(math.pi), (0.3, 0.2, 1.0)))             # fija: T_0C de S16
    a.pon('mesa', 'pieza', T(np.eye(3), (0.4, 0.2, 0.0)))                # la pieza de S17
    return a

In [ ]:
# Tres giros en el orden de ROS, frente al orden (s; v) de S8, y el rumbo de vuelta
for nombre, R in (('giro de 90° en z', Rz(math.pi / 2)), ('la mesa, Rz(−90°)', Rz(-math.pi / 2)),
                  ('la cámara cenital de S16, Rx(180°)', Rx(math.pi)), ('el robot en odom, a 88°', Rz(math.radians(88)))):
    q = cuaternion(R)
    print(f'{nombre:36s} ROS (x, y, z, w) = ({"; ".join(coma(v, 4) for v in q)})   S8 (s; v) = ({coma(q[3], 4)}; {", ".join(coma(v, 4) for v in q[:3])})')
q = (0.0, 0.0, 0.628, 0.779)          # lo que dijo tf2_echo al llegar a la meta de la ronda 2, en una prueba
print(f'\nel rumbo de (0; 0; 0,628; 0,779): θ = 2·atan2(z, w) = {coma(math.degrees(guinada(q)), 2)}°')

## 2. El árbol del almacén (REP 105) y el puesto de picking

*En clase, ejecuta esta celda sin detenerte.*

`arbol_almacen()` monta el árbol de la foto del robot aparcado en el picking, en (8,3; 8,2) y mirando a 90°:

- **map → odom**, la localización (en el taller la publica el simulador, que hace de AMCL): la deriva de un viaje, (0,30; −0,15) m y 2°;
- **odom → base_link**, la odometría: continua, con deriva;
- **base_link → laser**, el montaje del LiDAR, fijo;
- **map → mesa**, el puesto: Trans(8,3; 8,9; 0,75)·Rz(−90°), la mesa volada sobre la pared a 0,75 m del suelo;
- **mesa → camara**, la T_0C de S16: Trans(0,3; 0,2; 1,0)·Rx(180°), la nominal;
- **mesa → pieza**, lo que publicaría el detector de S17: (0,4; 0,2; 0).

Cada marco tiene un solo padre, así que entre dos marcos hay un solo camino. Las consultas son las de las diapositivas: map → laser = (8,3; 8,3; 0,2), base_link → pieza = (0,30; −0,20; 0,75) y camara → pieza = (0,1; 0; 1,0). Al final, el árbol sin map → odom: quedan dos árboles sueltos y la consulta falla, como `tf2_echo map base_link` con «not part of the same tree».

In [ ]:
A = arbol_almacen()
print('Las aristas (padre -> hijo):')
for padre, hijo in (('map', 'odom'), ('odom', 'base_link'), ('base_link', 'laser'), ('map', 'mesa'), ('mesa', 'camara'), ('mesa', 'pieza')):
    print(f'  {padre:9s} -> {hijo:9s} {describe(A.padre[hijo][1])}')
print('Las consultas (lookup_transform(destino, origen)):')
for destino, origen in (('map', 'base_link'), ('map', 'laser'), ('base_link', 'pieza'), ('camara', 'pieza')):
    print(f'  {destino:9s} -> {origen:9s} {describe(A.lookup(destino, origen))}')

# La cadena a mano: la letra de en medio se cancela
T_map_laser = A.padre['odom'][1] @ A.padre['base_link'][1] @ A.padre['laser'][1]
print('\nT_map,laser = T_map,odom · T_odom,base_link · T_base_link,laser:')
print(como_tf2_echo(T_map_laser))

# Sin map -> odom (el simulador con localizar:=false): dos árboles sueltos
suelto = Arbol()
for hijo in ('base_link', 'laser', 'mesa', 'camara', 'pieza'):
    suelto.pon(A.padre[hijo][0], hijo, A.padre[hijo][1])
try:
    suelto.lookup('map', 'base_link')
except LookupError as e:
    print('\nSin map -> odom:', e)

## 3. En clase: predecir lo que dirá `tf2_echo map laser`

*Es la parte de la ronda 2 que se hace aquí: unos dos minutos.*

Con el robot parado al acabar un viaje de la ronda 2, en una terminal del portátil (con `source install/setup.bash` hecho):

```
ros2 run tf2_ros tf2_echo map odom
ros2 run tf2_ros tf2_echo odom base_link
```

Copia en la celda la traslación (`Translation`) y el cuaternión (`Rotation: in Quaternion (xyzw)`) de cada una y ejecútala: escribe la predicción de map → laser en el mismo formato. Después, en la terminal, `ros2 run tf2_ros tf2_echo map laser`, y compara.

Los valores que hay en la celda son los de una prueba en ROS 2 Jazzy. Con ellos, la predicción sale `[1.819, 3.677, 0.200]` y `tf2_echo map laser` dijo `[1.817, 3.679, 0.200]`: unos 3 mm de diferencia, porque `tf2_echo` redondea el cuaternión a tres decimales, y en map → odom, con w = 1,000, una milésima de z es algo más de una décima de grado (2·0,001 rad = 0,11°): el redondeo, de hasta media milésima, mueve un punto a 4 m del origen del mapa hasta unos 4 mm. Con `-p 5` al final de las dos órdenes, `tf2_echo` escribe cinco decimales y la predicción coincide al milímetro.

In [ ]:
# Sección 3 · Copiad aquí lo que escriben tf2_echo map odom y tf2_echo odom base_link (con punto decimal, como en la consola)
MAP_ODOM  = dict(t=[-0.085, -0.034, 0.000], q=[0.000, 0.000, -0.018, 1.000])
ODOM_BASE = dict(t=[1.751, 3.679, 0.000], q=[0.000, 0.000, 0.641, 0.767])

def de_tf2_echo(t, q):
    """La transformada 4 x 4 de lo que escribe tf2_echo: la traslación t y el cuaternión q = (x, y, z, w)."""
    x, y, z, w = np.array(q, float) / np.linalg.norm(q)          # tf2_echo redondea: se vuelve a normalizar
    R = np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                  [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                  [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])
    return T(R, t)

pred = de_tf2_echo(**MAP_ODOM) @ de_tf2_echo(**ODOM_BASE) @ T(np.eye(3), LASER)
print('La predicción de tf2_echo map laser:')
print(como_tf2_echo(pred))
print(f'(el robot en map, con rumbo {coma(math.degrees(guinada(cuaternion(pred[:3, :3]))), 2)}°)')

## 4. odom frente a map: la ronda 2, sin ROS

La celda siguiente es el simulador del taller sin ROS, paso a paso como el nodo `simulador`: integra a 20 Hz (la pose verdadera con el ruido de la odometría de B6; la odometría, con la consigna sin ruido), publica a 10 Hz, corrige map → odom cada segundo (en cada corrección, map → base_link pasa a ser la verdad: una localización perfecta que hace de AMCL) y para la base si toca algo. `ronda2()` es el nodo `ir_a_punto`: cada 0,1 s lee la pose en `odom` o en `map` con lo último publicado y manda la ley de Corke (2023), v = K_v·d y ω = K_h·(atan2(Δy, Δx) − θ), con K_v = 0,5 y K_h = 1,5, saturadas a 0,5 m/s y 1,0 rad/s, hasta 2 cm de la meta.

`ronda1()`, el vigilante de la ronda 1, también está: es la que dio la distancia de parada de la puesta en común.

In [ ]:
# Las dos rondas del taller con el mismo simulador que el nodo de ROS 2 (20 Hz; publica a 10 Hz).
# =====================================================================================================
# S21 · Las dos rondas del taller, con el mismo simulador que el nodo de ROS 2 (20 Hz; publica a 10 Hz)
# =====================================================================================================
def se2(x, y, th):
    c, s = math.cos(th), math.sin(th)
    return np.array([[c, -s, x], [s, c, y], [0.0, 0.0, 1.0]])


def de_se2(M):
    return np.array([M[0, 2], M[1, 2], math.atan2(M[1, 0], M[0, 0])])


class Simulador:
    """El nodo simulador de curso_almacen sin ROS: verdad, odometría, map -> odom, choques y barridos."""

    def __init__(self, x0=0.6, y0=0.6, th0=0.0, semilla=0, localizar_cada=1.0, ruido=True, mundo=None):
        self.rng = np.random.default_rng(semilla)
        self.mundo = mundo or Mundo('abierta')
        self.verdad = np.array([x0, y0, th0])
        self.odo = self.verdad.copy()
        self.M = np.eye(3)                       # map -> odom
        self.cada, self.ruido = localizar_cada, ruido
        self.k, self.t_corr, self.choques = 0, localizar_cada, 0
        self.v = self.w = 0.0
        self.saltos = []                          # cuánto se mueve map -> base_link en cada corrección

    def paso(self, dt=0.05):
        """Un paso de 50 ms. Devuelve True si toca publicar (10 Hz)."""
        ds, dth = self.v * dt, self.w * dt
        nd = nt = 0.0
        if self.ruido and ds != 0.0:
            nd = self.rng.normal(0, SIG_D * math.sqrt(abs(ds)))
            nt = self.rng.normal(0, SIG_T * math.sqrt(abs(ds)))
        x, y, th = self.verdad
        nueva = np.array([x + (ds + nd) * math.cos(th), y + (ds + nd) * math.sin(th), th + dth + nt])
        if self.mundo.holgura(nueva[0], nueva[1]) < R_ROBOT and self.mundo.holgura(nueva[0], nueva[1]) < self.mundo.holgura(x, y):
            self.choques += 1
            self.v = self.w = 0.0
        else:
            self.verdad = nueva
            xo, yo, tho = self.odo
            self.odo = np.array([xo + ds * math.cos(tho), yo + ds * math.sin(tho), tho + dth])
        self.k += 1
        if self.k % 2:
            return False
        if self.k * dt >= self.t_corr - 1e-9:     # la localización corrige: map -> base_link = la verdad
            nuevo = se2(*self.verdad) @ np.linalg.inv(se2(*self.odo))
            p = np.array([self.odo[0], self.odo[1], 1.0])
            self.saltos.append(float(np.hypot(*((nuevo - self.M) @ p)[:2])))
            self.M = nuevo
            self.t_corr += self.cada
        return True

    def en_map(self):
        """map -> base_link, como lo compone TF2: map -> odom · odom -> base_link."""
        return de_se2(self.M @ se2(*self.odo))

    def barrido(self):
        x, y, th = self.verdad
        return self.mundo.barrido(x + LASER[0] * math.cos(th), y + LASER[0] * math.sin(th), th, self.rng, self.ruido)


ANG_HACES = ANG_MIN + INC * np.arange(N_HACES)
CONO = np.abs(ANG_HACES) <= math.radians(30.0) + 1e-9      # los 241 haces de −30° a +30°


def ronda1(semilla=0, v=0.3, umbral=0.5, t_max=15.0, mundo=None):
    """El vigilante: avanza a v y para si lo más cercano en ±30° baja de umbral. La orden se calcula al llegar cada
    barrido (10 Hz) y el simulador la aplica en su paso siguiente; sin pestillo: si el mínimo vuelve a pasar del umbral,
    avanza otro barrido (y escribe otro «Parado»). Devuelve el mínimo que lo paró la primera vez, la distancia de verdad
    del LiDAR a la estantería (x = 4,0 m) a los t_max segundos, el instante de la primera parada, los «Parado» escritos
    y los choques; y, al final, la distancia de verdad del LiDAR a la estantería en el barrido que lo paró."""
    sim = Simulador(0.6, 2.4, 0.0, semilla, mundo=mundo)
    minimo_parada, t_parada, paradas, t, d_parada = None, None, 0, 0.0, None
    while t < t_max:
        if sim.paso():
            r = sim.barrido()[CONO]
            r = r[np.isfinite(r)]
            minimo = r.min() if r.size else math.inf
            antes = sim.v
            sim.v = 0.0 if minimo < umbral else v
            if sim.v == 0.0 and antes != 0.0:
                paradas += 1
                if minimo_parada is None:
                    minimo_parada, t_parada = minimo, t
                    d_parada = 4.0 - (sim.verdad[0] + LASER[0] * math.cos(sim.verdad[2]))
        t += 0.05
    hueco = 4.0 - (sim.verdad[0] + LASER[0] * math.cos(sim.verdad[2]))
    return minimo_parada, hueco, t_parada, paradas, sim.choques, d_parada


def ronda2(semilla=0, marco='odom', meta=(1.8, 3.6), K_v=0.5, K_h=1.5, tol=0.02, cada=1.0, traza=False, mundo=None):
    """Ir a un punto: el controlador lee la pose (odom -> base_link, o map -> base_link) a 10 Hz, con lo último que
    publicó el simulador, y manda la ley de Corke (2023). Devuelve el error de llegada frente a la verdad, la
    deriva de la odometría al llegar (lo que separa map -> base_link de odom -> base_link), los saltos de cada
    corrección (en la pose del robot) y el tiempo; con traza, también las series."""
    sim = Simulador(0.6, 0.6, 0.0, semilla, localizar_cada=cada, mundo=mundo)
    est = sim.odo.copy() if marco == 'odom' else sim.en_map()
    serie, t = [], 0.0
    while t < 60:
        d = math.hypot(meta[0] - est[0], meta[1] - est[1])
        if d < tol:
            break
        e = (math.atan2(meta[1] - est[1], meta[0] - est[0]) - est[2] + math.pi) % (2 * math.pi) - math.pi
        sim.v, sim.w = min(K_v * d, V_MAX), max(-W_MAX, min(W_MAX, K_h * e))
        for _ in range(2):                        # dos pasos del simulador por ciclo del controlador (10 Hz)
            publica = sim.paso()
            t += 0.05
            if publica:                           # lo que lee el controlador: lo último publicado
                ahora = sim.en_map()
                est = sim.odo.copy() if marco == 'odom' else ahora
                if traza:
                    serie.append((t, *sim.verdad, *sim.odo, *ahora))
    err = math.hypot(meta[0] - sim.verdad[0], meta[1] - sim.verdad[1])
    deriva = math.hypot(*(sim.verdad[:2] - sim.odo[:2]))
    out = dict(error=err, deriva=deriva, saltos=np.array(sim.saltos), t=t, verdad=sim.verdad.copy(), odo=sim.odo.copy(),
               map_odom=de_se2(sim.M))
    if traza:
        out['serie'] = np.array(serie)
    return out

Del muelle a (1,8; 3,6), 3,23 m, con cincuenta semillas. A la izquierda, un viaje controlando en map (la semilla 1, la de la figura de las diapositivas): cuánto se separa de la verdad la pose contada en odom, en ámbar, y en map, en azul, con un diente de sierra que vuelve a cero en cada corrección. A la derecha, el error de llegada de las cincuenta semillas en los dos marcos. **En odom, el error de llegada frente a la verdad es la deriva de la odometría**: el robot llega a donde cree que está la meta, y la odometría se ha separado de la verdad lo que haya derivado en 3 m. **En map, es la tolerancia de 2 cm más lo que deriva en el último segundo**, porque cada corrección devuelve la pose a la verdad. `SEMILLA` es la de vuestro equipo: la del taller no sale igual, porque el ritmo de los dos procesos de ROS 2 cambia el orden de los sorteos; lo que se parece es el reparto de las cincuenta semillas, no la cifra de una.

In [ ]:
MUNDO = Mundo('abierta')
SEMILLA = 3                                   # la de vuestro equipo (semilla:= del almacén en el taller)
t0 = time.time()
res = {m: [ronda2(s, m, mundo=MUNDO) for s in range(50)] for m in ('odom', 'map')}
for m in ('odom', 'map'):
    e = np.array([o['error'] for o in res[m]]) * 100
    print(f'control en {m:4s}: error de llegada frente a la verdad, mediana {coma(np.median(e), 1)} cm, '
          f'p90 {coma(np.percentile(e, 90), 1)}, máx. {coma(e.max(), 1)} ; tarda {coma(np.median([o["t"] for o in res[m]]), 1)} s')
saltos = np.concatenate([o['saltos'] for o in res['map']]) * 100
print(f'saltos de map -> base_link en cada corrección: mediana {coma(np.median(saltos), 1)} cm, máx. {coma(saltos.max(), 1)}')
for m in ('odom', 'map'):
    o = ronda2(SEMILLA, m, mundo=MUNDO)
    print(f'semilla {SEMILLA}, en {m:4s}: error {coma(o["error"] * 100, 1)} cm ; la odometría se ha separado {coma(o["deriva"] * 100, 1)} cm de map')
print(f'({coma(time.time() - t0, 1)} s)')

o = ronda2(1, 'map', traza=True, mundo=MUNDO)        # la semilla 1, la de la figura de las diapositivas
s = o['serie']
t, ver, odo, mp = s[:, 0], s[:, 1:4], s[:, 4:7], s[:, 7:10]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={'width_ratios': [1.6, 1]})
a1.plot(t, np.hypot(*(odo[:, :2] - ver[:, :2]).T) * 100, color=AMBAR, lw=2.4, label='en odom: continua, con deriva')
a1.step(t, np.hypot(*(mp[:, :2] - ver[:, :2]).T) * 100, where='post', color=IQS_AZUL, lw=2, label='en map: corrige cada segundo')
a1.set_xlabel('tiempo (s)'); a1.set_ylabel('distancia a la verdad (cm)'); a1.set_title('semilla 1, controlando en map')
a1.legend(frameon=False); a1.grid(alpha=0.3)
for k, (m, c) in enumerate((('odom', AMBAR), ('map', IQS_AZUL))):
    e = np.array([x['error'] for x in res[m]]) * 100
    a2.scatter(np.full(e.size, k) + np.random.default_rng(k).uniform(-0.15, 0.15, e.size), e, s=14, color=c)
    a2.hlines(np.median(e), k - 0.3, k + 0.3, color='k', lw=2)
a2.set_xticks([0, 1], ['odom', 'map']); a2.set_ylabel('error de llegada (cm)'); a2.set_title('50 semillas'); a2.grid(alpha=0.3)
ejes(a1); a2.yaxis.set_major_formatter(COMA); plt.tight_layout(); plt.show()

## 5. Preguntar dónde estaba, no dónde está

TF2 guarda para cada arista las poses con su sello de tiempo, los últimos 10 s, y `lookup_transform(destino, origen, instante)` responde por un instante. Pedir un instante sin datos da `ExtrapolationException`: «al futuro» si el dato aún no ha llegado, «al pasado» si ya se borró. Mejor fallar que inventar.

La clase `Historia` hace lo mismo para una arista, con una simplificación: devuelve la última pose con sello anterior al pedido, **sin interpolar** (TF2 sí interpola entre las dos más cercanas: la interpolación es el ejercicio 3).

El ejemplo: el robot gira sin moverse en (5,0; 5,3), en la nave de arriba, a 0,5 rad/s, y TF publica su pose a 10 Hz. Llega un barrido con el sello t = 11,88 s, cuando el robot miraba a 0°, y el código lo procesa 120 ms después, a los 12,00 s. Colocarlo con la última pose del buffer (la de 12,00 s) lo tuerce ω·Δt = 0,5 · 0,12 = 0,06 rad = 3,44°: a 8 m, 48 cm. Preguntar por su sello lo arregla; sin interpolar, a medias.

In [ ]:
class Extrapolacion(Exception):
    pass


class Historia:
    """Lo que guarda el buffer de TF2 para una arista: las poses (x; y; θ) con su sello, los últimos 10 s.
    en(t) da la última pose con sello <= t, sin interpolar; fuera de lo guardado, Extrapolacion."""

    def __init__(self, dura=10.0):
        self.t, self.p, self.dura = [], [], dura

    def pon(self, t, pose):
        self.t.append(t); self.p.append(np.array(pose, float))
        while self.t[-1] - self.t[0] > self.dura + 1e-9:
            self.t.pop(0); self.p.pop(0)

    def en(self, t):
        if t > self.t[-1] + 1e-9:
            raise Extrapolacion(f'pedir t = {coma(t)} s requeriría extrapolar al futuro: el último dato es de t = {coma(self.t[-1])} s')
        if t < self.t[0] - 1e-9:
            raise Extrapolacion(f'pedir t = {coma(t)} s requeriría extrapolar al pasado: el primero es de t = {coma(self.t[0])} s')
        return self.p[int(np.searchsorted(self.t, t + 1e-9)) - 1]


W, T_MED, X0, Y0 = 0.5, 11.88, 5.0, 5.3            # gira a 0,5 rad/s; el barrido lleva el sello de 11,88 s
pose = lambda t: (X0, Y0, W * (t - T_MED))          # a los 11,88 s mira a 0°
H = Historia()
for k in range(0, 121):                              # TF a 10 Hz durante 12 s: el buffer guarda de 2,0 a 12,0
    H.pon(k / 10, pose(k / 10))
for pedido in (12.05, 1.0):
    try:
        H.en(pedido)
    except Extrapolacion as e:
        print('Extrapolacion:', e)

barrido = MUNDO.barrido(X0 + LASER[0], Y0, 0.0, np.random.default_rng(1), ruido=False)   # lo que midió, a los 11,88 s
ok = np.isfinite(barrido)
def coloca(th):
    """Los puntos del barrido en map, con el robot en (X0; Y0) y rumbo th (el LiDAR, 10 cm por delante)."""
    lx, ly = X0 + LASER[0] * math.cos(th), Y0 + LASER[0] * math.sin(th)
    return lx + barrido[ok] * np.cos(th + ANG_HACES[ok]), ly + barrido[ok] * np.sin(th + ANG_HACES[ok])
bueno = coloca(pose(T_MED)[2])
for nombre, th in (('con la última pose (12,00 s)', H.en(12.0)[2]), ('con su sello, sin interpolar', H.en(T_MED)[2]),
                   ('con su sello, interpolando', pose(T_MED)[2])):
    malo = coloca(th)
    d = np.hypot(malo[0] - bueno[0], malo[1] - bueno[1]) * 100
    print(f'{nombre:30s} el rumbo sale {coma(math.degrees(th - pose(T_MED)[2]), 2)}° desviado ; '
          f'los puntos se mueven {coma(np.median(d), 0)} cm de mediana y {coma(d.max(), 0)} el más lejano (a {coma(barrido[ok].max(), 1)} m)')

occ = MUNDO.occ
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.imshow(occ, origin='lower', extent=(0, COLS * RES, 0, FILAS * RES), cmap='Greys', vmin=0, vmax=1.6)
ax.scatter(*bueno, s=3, color=IQS_VERDE, label='con la pose de su sello (11,88 s)')
ax.scatter(*coloca(H.en(12.0)[2]), s=3, color=ROJO, label='con la última pose (12,00 s): 3,44° más')
ax.plot(X0, Y0, 'o', color=IQS_AZUL, ms=9)
ax.set_xlim(0, 13); ax.set_ylim(4.0, 9.0); ax.set_aspect('equal'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.legend(loc='lower right', markerscale=5, fontsize=9); ejes(ax); plt.tight_layout(); plt.show()

## 6. La cámara de S16 y la pieza, en el mismo árbol (para la opción A)

La arista mesa → camara es la T_0C de vuestra calibración de S16 (la que use el equipo en la entrega 3). El fichero `calibracion_E⟨n⟩⟨letra⟩.json` guarda la T_C0 (la mesa vista desde la cámara): T_0C es su inversa. Para usar la vuestra, sube el fichero a la carpeta de Colab (el icono de la carpeta, a la izquierda) y escribe su nombre en `CALIBRACION`; sin fichero, la celda usa la nominal del plano del montaje y lo dice.

La celda escribe la orden de `static_transform_publisher` que publica vuestra cámara en el árbol, con el cuaternión en el orden de ROS, y las tres consultas del puesto. **base_link → pieza no depende de la cámara** cuando la pieza se da en la mesa; la cámara entra cuando el detector la da en su propio marco, p_C: entonces la pieza en base_link es T_base_link,mesa · T_mesa,camara · p_C, y una T_0C equivocada la mueve.

Al final, el robot aparcado 5 cm y 2° fuera de la meta: si se supone que está justo en el picking, la pieza se equivoca en 6 cm. Por eso se pregunta al árbol con la pose localizada, y no con la meta.

In [ ]:
CALIBRACION = 'calibracion_E0A.json'          # el fichero de vuestra calibración de S16
T_0C_NOMINAL = T(Rx(math.pi), (0.3, 0.2, 1.0))  # lo que dice el plano del montaje
if os.path.exists(CALIBRACION):
    T_0C = inv(np.array(json.load(open(CALIBRACION, encoding='utf-8'))['T_C0'], float))
    print(f'T_0C de {CALIBRACION}:', describe(T_0C))
else:
    T_0C = T_0C_NOMINAL
    print(f'No encuentro {CALIBRACION}: uso la T_0C nominal,', describe(T_0C))

A = arbol_almacen()
A.pon('mesa', 'camara', T_0C)
t_, q_ = T_0C[:3, 3], cuaternion(T_0C[:3, :3])
print('\nLa orden que publica vuestra cámara en el árbol:')
print(f'ros2 run tf2_ros static_transform_publisher --x {t_[0]:.4f} --y {t_[1]:.4f} --z {t_[2]:.4f} '
      f'--qx {q_[0]:.6f} --qy {q_[1]:.6f} --qz {q_[2]:.6f} --qw {q_[3]:.6f} --frame-id mesa --child-frame-id camara')
print()
for destino, origen in (('map', 'camara'), ('camara', 'pieza'), ('base_link', 'pieza')):
    print(f'{destino:9s} -> {origen:6s} {describe(A.lookup(destino, origen))}')

p_C = inv(T_0C_NOMINAL) @ np.array([0.4, 0.2, 0.0, 1.0])     # la pieza en el marco de la cámara, como la daría el detector
T_bl_mesa = A.lookup('base_link', 'mesa')
d = np.linalg.norm((T_bl_mesa @ T_0C @ p_C - T_bl_mesa @ T_0C_NOMINAL @ p_C)[:3]) * 1000
print(f'\nla pieza en base_link desde p_C, con vuestra T_0C frente a la nominal: {coma(d, 1)} mm de diferencia')

Ar = arbol_almacen(robot_map=(8.3, 8.25, math.radians(92)))     # aparcado 5 cm y 2° fuera de la meta
b = Ar.lookup('base_link', 'pieza')[:3, 3]
print(f'robot 5 cm y 2° fuera de la meta: base_link -> pieza = ({coma(b[0])}; {coma(b[1])}; {coma(b[2])}) m ; '
      f'suponerlo en la meta equivoca la pieza en {coma(np.hypot(b[0] - 0.30, b[1] + 0.20) * 100, 1)} cm')

### Ejercicio 1 (en casa): el cuaternión en el orden de S8

En la ronda 2, el hueco 2 de `ir_a_punto.py` es el rumbo desde el cuaternión, θ = 2·atan2(z, w). Un compañero piensa en el orden (s; v) de S8 y escribe `th = 2 * math.atan2(q.w, q.x)`, como si `q.x` fuera el escalar. (a) ¿Qué rumbo lee para el robot a 0°, a 88° y a −90°? (b) ¿Por qué su robot gira sin parar? (c) Y al revés: si al publicar se copia el (s; v) de S8 en los campos (x, y, z, w), ¿qué giro recibe TF2 para el robot a 88°? Usa `de_tf2_echo()` de la sección 3 y mira la matriz.

In [ ]:
# Ejercicio 1
# for grados in (0, 88, -90):
#     x, y, z, w = cuaternion(Rz(math.radians(grados)))
#     print(grados, math.degrees(2 * math.atan2(w, x)))
# s, vx, vy, vz = ...                                  # el (s; v) de S8 del robot a 88°
# R = de_tf2_echo([0, 0, 0], [s, vx, vy, vz])[:3, :3]  # copiado en (x, y, z, w)

### Ejercicio 2 (en casa): corregir más o menos a menudo

El simulador corrige map → odom cada segundo (`localizar_cada` en el taller; `cada` en `ronda2`). Con las cincuenta semillas de la sección 4, controlando en map, prueba a corregir cada 0,5, 2 y 5 s. ¿Cómo cambian el error de llegada y el mayor salto de map → base_link? ¿Qué se vería en RViz?

In [ ]:
# Ejercicio 2
# for cada in (0.5, 1.0, 2.0, 5.0):
#     o = [ronda2(s, 'map', cada=cada, mundo=MUNDO) for s in range(50)]
#     ...

### Ejercicio 3 (en casa): la interpolación en el tiempo, a mano

TF2 no da la última pose anterior al instante pedido: interpola entre las dos que lo rodean. Escribe `en_interpolando(H, t)`: busca las dos muestras que rodean t, calcula la fracción a = (t − t_k)/(t_k+1 − t_k) e interpola x, y y θ (el ángulo, por la diferencia más corta, envuelta en (−π, π]). Con la `Historia` de la sección 5, pide cien instantes repartidos entre 11,00 y 11,99 s y compara el error de rumbo frente a la pose verdadera con `H.en()` y con tu función. ¿Cuánto se mueve, como mucho, un punto a 8 m con cada una?

In [ ]:
# Ejercicio 3
# def en_interpolando(H, t):
#     k = int(np.searchsorted(H.t, t)) - 1          # H.t[k] <= t < H.t[k + 1]
#     ...
# instantes = np.linspace(11.0, 11.99, 100)

---

## Soluciones

**Ejercicio 1.** (a) Lee 180° para el robot a 0°, a 88° y a −90°: con un giro solo en z, `q.x` vale 0 y `q.w` = cos(θ/2) es positivo, así que atan2(q.w, 0) = 90° siempre, y el doble, 180°. (b) El rumbo leído no cambia aunque el robot gire, así que el error de rumbo no se cierra nunca: ω se queda saturada en 1,0 rad/s y el robot gira sin parar (es uno de los fallos típicos de la ronda 2). (c) El robot a 88° tiene (s; v) = (0,7193; 0, 0, 0,6947); copiado en (x, y, z, w) es (0,7193; 0; 0; 0,6947): un giro de 2·acos(0,6947) = 92,0° alrededor de x. TF2 cree que el robot está volcado de lado y no ha girado nada en z: el rumbo que da 2·atan2(z, w) es 0°.

```python
for grados in (0, 88, -90):
    x, y, z, w = cuaternion(Rz(math.radians(grados)))
    print(grados, coma(math.degrees(2 * math.atan2(w, x)), 2))        # 180,00 las tres veces
x, y, z, w = cuaternion(Rz(math.radians(88)))
R = de_tf2_echo([0, 0, 0], [w, x, y, z])[:3, :3]                     # (s; v) copiado en (x, y, z, w)
print(np.round(R, 4), coma(2 * math.degrees(math.acos(z)), 1))       # Rx(92,0°): w pasa a ser la z de antes
```

**Ejercicio 2.** Controlando en map, con las cincuenta semillas: corrigiendo cada 0,5 s, error de llegada de 1,8 cm de mediana (p90 2,1) y saltos de hasta 9,0 cm; cada 1 s, 1,8 cm (p90 2,2) y hasta 11,7; cada 2 s, 1,6 cm (p90 2,7) y hasta 16,4; cada 5 s, 1,9 cm (p90 3,4) y hasta 20,7. El error de llegada apenas cambia, porque el último tramo es lento y casi no deriva; lo que crece es el salto: en RViz, el robot salta hasta 21 cm cada vez que llega una corrección (los dientes de sierra de la sección 4, más grandes y más espaciados).

```python
for cada in (0.5, 1.0, 2.0, 5.0):
    o = [ronda2(s, 'map', cada=cada, mundo=MUNDO) for s in range(50)]
    e = np.array([x['error'] for x in o]) * 100
    sl = np.concatenate([x['saltos'] for x in o]) * 100
    print(coma(cada, 1), coma(np.median(e), 1), coma(np.percentile(e, 90), 1), coma(sl.max(), 1))
```

**Ejercicio 3.** Con `H.en()`, el error de rumbo es lo que el robot ha girado desde la última muestra: con los cien instantes, cada 0,01 s, de 0 a 2,58°, y 1,29° de media (el máximo posible, justo antes de la muestra siguiente, es 0,5 rad/s por 0,1 s, 2,86°); un punto a 8 m se mueve hasta 36 cm. Interpolando, el error es cero (el robot gira a ritmo constante, y la interpolación lineal es exacta). En un robot real, el LiDAR y la odometría no publican a la vez, así que los sellos casi nunca caen en una muestra: por eso TF2 interpola.

```python
def en_interpolando(H, t):
    k = int(np.searchsorted(H.t, t)) - 1
    if k < 0 or k + 1 >= len(H.t):
        return H.en(t)
    a = (t - H.t[k]) / (H.t[k + 1] - H.t[k])
    p0, p1 = H.p[k], H.p[k + 1]
    dth = (p1[2] - p0[2] + math.pi) % (2 * math.pi) - math.pi
    return np.array([p0[0] + a * (p1[0] - p0[0]), p0[1] + a * (p1[1] - p0[1]), p0[2] + a * dth])

instantes = np.linspace(11.0, 11.99, 100)
e_en = np.array([abs(H.en(t)[2] - pose(t)[2]) for t in instantes])
e_in = np.array([abs(en_interpolando(H, t)[2] - pose(t)[2]) for t in instantes])
print(coma(np.degrees(e_en.max()), 2), coma(np.degrees(e_en.mean()), 2), coma(np.degrees(e_in.max()), 2), coma(8 * np.tan(e_en.max()), 2))
```

---

## Para llevarse de esta sesión

- **T_AB es la pose de {B} vista desde {A}**, «A → B» en TF2; dos seguidas se multiplican y la letra de en medio se cancela.
- **El árbol de marcos tiene un padre por marco**: entre dos marcos hay un solo camino; sin una arista, quedan dos árboles y la consulta falla.
- **El cuaternión de ROS es (x, y, z, w) = (v; s)**, el de S8 en otro orden; el rumbo de un giro en z es 2·atan2(z, w).
- **REP 105: odom es continua y deriva; map no deriva y salta.** Del muelle a (1,8; 3,6), en odom se llega con 9,3 cm de error de mediana y en map con 1,8: se mueve en odom y se llega en map.
- **Se pregunta por el instante de la medida**, el sello del mensaje: con la pose de 120 ms después, girando a 0,5 rad/s, un punto a 8 m se va 48 cm. TF2 interpola, y prefiere fallar a extrapolar.
- **La cámara y la pieza son aristas fijas del mismo árbol**: con el robot en el picking, base_link → pieza = (0,30; −0,20; 0,75) m.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*